In [1]:

from pyspark.sql import *
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("Spark Optimization") \
.config("spark.sql.ui.explainMode", "extended").getOrCreate()

In [2]:
df = spark.read.format("csv").option("header", True).option("inferSchema", True)\
.load("BigMart Sales.csv")
df.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

In [17]:
df.select("Item_Identifier").distinct().show()

+---------------+
|Item_Identifier|
+---------------+
|          FDB11|
|          FDO11|
|          DRA24|
|          FDU24|
|          FDW60|
|          FDW52|
|          FDN15|
|          FDQ20|
|          FDU10|
|          FDY43|
|          NCL54|
|          FDP39|
|          NCN30|
|          NCS30|
|          NCW05|
|          NCF42|
|          FDX27|
|          FDD20|
|          FDW32|
|          NCW18|
+---------------+
only showing top 20 rows


# partitioned data 

In [10]:
df.write.format("parquet").mode("append")\
.partitionBy("Item_Identifier")\
.option("path", "outputData/partitioned-Item_Identifier").save()

# non partitioned Data 

In [11]:
df.write.format("parquet").mode("append")\
.option("path", "outputData/non-partitioned").save()

# read data

In [11]:
df1 = spark.read.format("parquet")\
.option("path", "outputData/partitioned-Item_Identifier").load()
df1.rdd.getNumPartitions()

49

In [12]:
df2 = spark.read.format("parquet")\
.option("path", "outputData/non-partitioned").load()
df2.rdd.getNumPartitions()

1

# set up spark config

In [24]:
spark.conf.set("spark.sql.adaptive.enabled", "false") # Enable AQE
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true") # Adjust DPP
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 5*1024*1024)  # Disable broadcast joins

# JOIN

In [25]:
df_join = df1.join(df2.filter(col("Item_Identifier") == "NCW18"), df1["Item_Identifier"] == df2["Item_Identifier"], "inner" )
# df_join.explain(extended=True)
df_join.count()

16

# check http://localhost:4040/